# tinylm sft — analysis

Analysis-only notebook (parallels `../pretrain/main.ipynb`): loads **one** SFT
checkpoint produced by `train.py`, selected in the setup cell, and runs chat
generations, per-source masked val loss, and zero-shot benchmarks (SFT shouldn't
tank them) on it. Change `CKPT`/`SEQ_LEN` in the setup cell to test a different
checkpoint. Run `train.py` first.

In [ ]:
import os

os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")
# SFT checkpoints were trained on the pinned 16k tokenizer; pin it before
# importing train so dm.tokenizer matches the checkpoint — a mismatched vocab
# decodes to garbage. Mirrors the pretrain notebook.
os.environ.setdefault(
    "TINYLM_TOKENIZER_PATH",
    "/root/Code/chimera/projects/tinylm/data/tokenizers/16k/tokenizer.json",
)

from pathlib import Path

import torch

import train  # sft/train.py: constants, make_model, make_datamodules

DEVICE = train.DEVICE
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

In [ ]:
# Build the SFT datamodules once (pinned vocab) for the tokenizer + per-source
# val loaders. Parallels the pretrain notebook's make_datamodule cell.
dms = train.make_datamodules()
tok = dms[0].tokenizer
eos_id, bos_id = dms[0].eos_id, dms[0].bos_id
im_end_id = tok._tok.token_to_id("<|im_end|>")

print(f"vocab_size={tok.vocab_size}  eos={eos_id} bos={bos_id} im_end={im_end_id}")
for dm in dms:
    print(f"  {dm.name}: {len(dm.train_dataset.ids):,} train tokens")

In [ ]:
# Pick which SFT checkpoint to test (see sft/README Results). Set SEQ_LEN to the
# base's training context — RoPE is computed on the fly so any value loads, but
# matching it lets long grounded prompts fit. SFT checkpoints are plain VWN(1,1).
CKPT = train.RUN_DIR / "chimera_gpt6m_sft_ctx4k.pt"  # best assistant so far
SEQ_LEN = 4096
# other options in train.RUN_DIR:
#   chimera_gpt6m_sft.pt           base-512 grounded SFT   (SEQ_LEN=512)
#   chimera_gpt6m_sft_ctx2k.pt     ctx2k grounded          (SEQ_LEN=2048)
#   chimera_gpt6m_sft_ctx8k.pt     ctx8k grounded          (SEQ_LEN=8192)
#   chimera_gpt6m_sft_ctx4k_p2.pt  ctx4k + breadth (Phase-2, SEQ_LEN=4096)
#   chimera_gpt6m_sft_lora.pt      LoRA                    (SEQ_LEN=512)
assert CKPT.exists(), f"run sft/train.py first: {CKPT} missing"

state = torch.load(CKPT, map_location="cpu")
# older checkpoints were saved from the torch.compile wrapper -> _orig_mod. prefixes
state = {k.removeprefix("_orig_mod."): v for k, v in state.items()}
train.SEQ_LEN = SEQ_LEN  # make_model reads this module global
model = train.make_model(tok.vocab_size)
model.load_state_dict(state)
model.to(DEVICE, DTYPE).eval()  # move explicitly — CPU inference is ~30x slower
print(
    f"loaded {CKPT.name}  (seq_len={SEQ_LEN}, "
    f"{sum(p.numel() for p in model.parameters()):,} params, device={DEVICE})"
)

## Masked val loss per source

Loss over assistant tokens only, per SFT source. `base` shows where each
checkpoint started; deltas show what SFT actually moved.


In [ ]:
import pandas as pd

# masked val loss (assistant tokens only) per source, for the loaded checkpoint
pd.Series(
    {dm.name: train.evaluate(model, dm.val_dataloader(), eos_id) for dm in dms},
    name="masked_val_loss",
).round(4)

## Chat


In [ ]:
from chimera.data.chat_template import render


@torch.no_grad()
def chat(question, max_new_tokens=80):
    prompt = render([{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tok._tok.encode(prompt, add_special_tokens=False).ids
    x = torch.tensor([[bos_id] + ids], device=DEVICE)
    out = []
    for _ in range(max_new_tokens):
        nxt = int(model(x)[0, -1].argmax())
        if nxt in (eos_id, im_end_id):  # stop at end-of-turn, not just EOS
            break
        out.append(nxt)
        x = torch.cat([x, torch.tensor([[nxt]], device=DEVICE)], dim=1)
    return tok._tok.decode(out).strip()


PROMPTS = [
    "What color is the sky?",
    "Hi! How are you today?",
    "Tom has a red ball and a blue kite. He plays with them in the park every "
    "Sunday with his sister Amy.\n\nWhat color is Tom's ball?",
    "Summarize this in one sentence: The library was founded in 1475 and holds "
    "ancient manuscripts, serving scholars of history and law worldwide.",
]

for q in PROMPTS:
    print(f"### {q[:70]}\n  {chat(q)!r}\n")

## Zero-shot benchmarks

Same task set as pretrain. The check is that SFT didn't tank the base
capabilities (catastrophic forgetting), not that it improves them.


In [ ]:
from chimera.evals import ChimeraLM, TASKS, headline, run_eval

lm = ChimeraLM(
    model,
    tok,
    eot_id=eos_id,
    bos_id=bos_id,
    block_size=model.seq_len,
    device=DEVICE,
    batch_tokens=131_072,
)
results = run_eval(lm, TASKS)
pd.Series(
    {t: headline(results[t])[1] * 100 for t in TASKS}, name="zero_shot_%"
).round(2)

Past results and run notes live in the [README](README.md).


## Playground

Free-form experimentation with the FULL sampler (`model.sample`): temperature,
top-p/top-k, min-p, repetition / frequency / presence penalties, seed. `generate()`
takes a plain string (single user turn) or a full `messages` list (multi-turn),
renders it through the canonical chat template, and returns only the assistant
continuation (sliced in token space). Tweak and re-run the last cell.


In [ ]:
def generate(
    prompt_or_messages,
    max_new_tokens=128,
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    min_p=0.0,
    repetition_penalty=1.1,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    seed=None,
):
    messages = (
        [{"role": "user", "content": prompt_or_messages}]
        if isinstance(prompt_or_messages, str)
        else prompt_or_messages
    )
    prompt = render(messages, add_generation_prompt=True)
    n_prompt = 1 + len(tok._tok.encode(prompt, add_special_tokens=False).ids)  # +bos
    _, generated = model.sample(
        tok,
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        min_p=min_p,
        repetition_penalty=repetition_penalty,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
        bos_token_id=bos_id,
        stop_token_ids={eos_id, im_end_id},
        seed=seed,
        return_token_ids=True,
    )
    token_ids = generated[0].tolist()  # sample returns a (1, T) tensor
    cont = [t for t in token_ids[n_prompt:] if t not in (eos_id, im_end_id)]
    return tok._tok.decode(cont).strip()

In [18]:
# sampler A/B on one prompt: greedy vs default vs hot + heavy anti-repetition
q = "What color is the sky?"
for label, kw in {
    "greedy": dict(temperature=0.0),
    "default": dict(seed=0),
    "hot+penalized": dict(
        temperature=1.0,
        repetition_penalty=1.3,
        frequency_penalty=1.1,
        presence_penalty=1.1,
        seed=0,
    ),
}.items():
    print(f"{label:>14}: {generate(q, **kw)!r}\n")

        greedy: "The sun is a bright blue, and it's a very bright yellow. The sun is light pink with white and has a red and yellow color. The sun is a bright yellow color because it's bright green and has a long-lived appearance."

       default: "Both colors are a lot of white. ... Black, blue and yellow are both common red and purple because they can be colored or yellow. It's pretty small, but has been suggested by the American Academy for more than half of our time and could also have made us black as long as we know it."

 hot+penalized: "Both colors are a lot of white. ... Black (because they're blue and dark) yellow, which can be colored or brown at some point to reddeny their lengths from black under light for grayness that aren't too close as green! They have little resembles orange because they just like pink on them – with shades similar in colour depending upon why you've left-handed it — but don's bright brighter after all..."



In [ ]:
# scratch cell — try your own prompt / knobs on the loaded `model`
print(
    generate(
        [{"role": "user", "content": "What is the capital of France?"}],
        temperature=0,
        repetition_penalty=1.1,
        seed=1,
        max_new_tokens=256,
    )
)